# ABLAÇÃO DO POOLER — Pair-Aware linear **com pooler** (`pair_aware_pooler`) BioBERTpt-all — RECLin-PT (Colab T4)

**Execução exploratória de ablação (Etapa 5, recomendação M3).** O Baseline e a Pair-Aware linear diferem em **duas** coisas ao mesmo tempo: a Pair-Aware lê os marcadores `[E1]`/`[E2]` **e** dispensa o pooler do encoder. Para separar os dois efeitos, a ablação treina as duas células que faltam do fatorial 2×2:

| | com pooler | sem pooler |
| --- | --- | --- |
| só o `[CLS]` | `baseline` (já treinado) | **`cls_direto`** |
| `[CLS]` + `[E1]` + `[E2]` | **`pair_aware_pooler`** | `pair_aware` (já treinada) |

**Este notebook treina a estratégia `pair_aware_pooler`**, que lê o `[CLS]` e os marcadores, como a Pair-Aware linear, mas o `[CLS]` **passa pelo pooler** (dense + tanh) do encoder, como no Baseline. Cabeça: `[pooler(h_cls) ; h_[E1] ; h_[E2]]` → Dropout → Linear(3d, 3). Todo o resto é o do protocolo (D10): `pucpr/biobertpt-all` na revisão fixada, seed 42, partições v2, `max_gap=25`, todos os pares candidatos (sem filtro lexical, sem léxico), `class_weight=balanced`, 3 épocas, `batch_size=64`, `lr=2e-5`, seleção da época pelo **macro-F1 do DEV completo**.

**Natureza.** A estratégia é posterior ao protocolo pré-registrado da fase: a execução é **EXPLORATÓRIA**, sem nome oficial, com pasta própria (`pair_aware_pooler__<config_id>__t1`), nunca usa nem sobrescreve as pastas do Baseline ou da Pair-Aware e **não é elegível ao TEST**. Ela serve para explicar de onde vem a diferença entre Baseline e Pair-Aware, não para escolher o modelo final.

**Só DEV.** Este notebook **não lê nem avalia o TEST**. A seção 5 confere a integridade das partições pelo SHA-256 do MANIFEST, como os demais notebooks; nenhum documento do TEST é carregado. A seção 8.3 calcula, no DEV, as diferenças pareadas desta execução contra o Baseline e a Pair-Aware (bootstrap por candidato e por documento).

**Código.** A execução registra a identidade do código clonado. As estratégias de ablação só existem a partir do commit que as inclui: **faça o commit e o push do código da Etapa 5 antes de abrir este notebook** (o clone usa o secret `GITHUB_PAT`, então funciona com o repositório privado).

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre os secrets `GITHUB_PAT` e `HF_TOKEN` (ícone de chave 🔑, com **Notebook access** ligado). O treino é feito no disco local e **espelhado a cada checkpoint no Google Drive** (`MyDrive/RECLin-PT/execucoes/`); se o runtime cair, **rode a seção 7 de novo**: ela retoma da execução interrompida compatível. Envios para fora (GitHub e Hugging Face) só acontecem nas seções 10 e 11, com confirmação explícita.

## 1. Clonar o repositório

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

try:
    userdata.get('GITHUB_PAT')               # secret do Colab, nunca hard-coded (o valor não é guardado nem impresso)
except Exception as erro:
    raise SystemExit('Secret GITHUB_PAT indisponível (%s). Colab: ícone de chave → "+ Add new secret" → nome '
                     'GITHUB_PAT → cole o token → ligue "Notebook access" → rode esta célula de novo.' % type(erro).__name__)
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

# O token NÃO entra na URL (ficaria gravado em .git/config): vai por GIT_ASKPASS (script 700, apagado ao
# terminar), só no ambiente do processo do git.
url = f'https://github.com/angeloalsf/{REPO_NAME}.git'

def git(*args, check=True):
    pasta = tempfile.mkdtemp(prefix='askpass-'); os.chmod(pasta, 0o700)
    script = os.path.join(pasta, 'askpass.sh')
    with open(script, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in *sername*) printf "x-access-token";; *) printf "%s" "$RECLIN_GIT_TOKEN";; esac\n')
    os.chmod(script, 0o700)
    env = {**os.environ, 'GIT_ASKPASS': script, 'GIT_TERMINAL_PROMPT': '0', 'RECLIN_GIT_TOKEN': userdata.get('GITHUB_PAT')}
    try:
        return subprocess.run(['git', '-c', 'credential.helper=', *args], env=env, check=check)
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

if not os.path.isdir(REPO):
    git('clone', url, REPO)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    git('-C', REPO, 'pull', '--rebase', '--autostash', check=False)

git('-C', REPO, 'config', 'user.name', GIT_USER_NAME)
git('-C', REPO, 'config', 'user.email', GIT_USER_EMAIL)
git('-C', REPO, 'remote', 'set-url', 'origin', url)

# Código novo do projeto: fica em REPO/codigo. Registra o commit clonado (HEAD) na proveniência de cada execução.
BRANCH = subprocess.run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO}/codigo')
from reclin.experimento import apresentacao, codigo_colab, operacoes, plano, raiz, segredos
INFO = codigo_colab.clonar_codigo(f'angeloalsf/{REPO_NAME}', BRANCH, REPO)
SHA_CODIGO = INFO['sha']
CODIGO = Path(INFO['codigo'])
raiz.ativar(CODIGO)                          # põe codigo/ no sys.path (devolve o ambiente de subprocessos, não o caminho)
print('Repo pronto em', REPO, f'| branch {BRANCH} | commit {SHA_CODIGO}')

## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Montar o Google Drive (checkpoints)

Execuções em `MyDrive/RECLin-PT/execucoes/<nome>/` — **pasta própria** de cada execução (o nome inclui a configuração). Dentro: `checkpoints/melhor_modelo/` (pesos da melhor época) e `checkpoints/ultimo.pt` (estado de retomada). O treino roda no disco local e o Drive recebe uma cópia verificada (SHA-256).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ_DRIVE = Path(f'/content/drive/MyDrive/{REPO_NAME}')
RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
RAIZ_LOCAL = Path('/content/reclin_resultados/execucoes')       # disco local do Colab
RAIZ_PESOS = Path('/content/reclin_pesos')
print('Execuções espelhadas em:', RAIZ_DRIVE)
existe = any(e['estado'] in ('interrompida', 'iniciada') for e in plano.execucoes(RAIZ_LOCAL, RAIZ_DRIVE))
print('execução interrompida existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')

## 4. Instalar dependências

**Só esta célula instala dependências** (nada é instalado ao abrir o notebook nem em outras seções). Fonte única: `codigo/requirements.txt` do repo — a mesma lista usada fora do Colab, com as versões **fixadas com `==`**.

O `pip` **não reinstala** o `torch` enquanto a imagem do Colab trouxer a versão fixada (`torch==2.11.0`; o `+cu128` da imagem é ignorado na comparação pela PEP 440), e o mesmo vale para os demais pins que já batem com a imagem, então esta célula é rápida.

⚠️ Isso vale **enquanto o Colab ficar no 2.11.0**. Se a imagem subir e esta célula começar a demorar (baixa o `torch` do PyPI, vários GB), recapture as versões e suba o pin, nunca volte para `>=`. Nunca escreva `torch==2.11.0+cu128`. Se o pip trocar um pacote já importado (por exemplo `numpy`), o Colab pede para **reiniciar a sessão**: reinicie e rode tudo de novo, do início.

In [ ]:
!pip install -q -r $REPO/codigo/requirements.txt

## 5. Conferir os splits e os pesos

As partições (`codigo/dados/particoes-v2/*.jsonl`, seed 42, nível de documento) são **versionadas e congeladas** no repo — **as mesmas de todas as estratégias**. Esta célula só **confere** (arquivos e SHA-256 contra o MANIFEST); **não baixa nem reconstrói** dados.

**Só a segunda célula desta seção baixa os pesos** (nenhuma outra seção acessa o Hub para baixar; o treino recusa se eles faltarem). Os pesos do encoder são baixados do Hub na **revisão fixada** (`REVISAO_PESOS`), com conferência dos hashes dos arquivos; o commit usado fica registrado e congelado para a fase.

In [ ]:
REVISAO_PESOS = '36b11cfdfa49876b108fdbe88eaeeb9be3b03f1c'   # commit de pucpr/biobertpt-all

def _espec():
    from reclin.estrategias.ablacao_pooler import ConfigAblacao, pair_aware_pooler  # importada na hora (depois da seção 4)
    # todos os pares, sem filtro lexical e sem léxico: a especificação não carrega nada além da configuração
    return operacoes.Especificacao(estrategia=pair_aware_pooler.NOME, script='treinar.py', config_cls=ConfigAblacao,
                                   estrategia_config=pair_aware_pooler.registro())

def _ctx(**kw):
    return apresentacao.contexto(CODIGO, RAIZ_LOCAL, RAIZ_DRIVE, RAIZ_PESOS, REVISAO_PESOS, _espec(), **kw)

def _ctx_exec(restaurar=False, **kw):
    # contexto da execução mais recente (local ou Drive); com restaurar=True traz do Drive para o disco local
    try:
        return _ctx(execucao=apresentacao.execucao_atual(_ctx(), restaurar=restaurar), **kw)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None

splits = os.path.join(REPO, 'codigo', 'dados', 'particoes-v2')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'MANIFEST.json']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    raise FileNotFoundError('Partições não encontradas em codigo/dados/particoes-v2/. Elas são versionadas e '
                            'congeladas; este notebook não baixa nem reconstrói dados. Confira a branch clonada.')

PART = apresentacao.conferir_particoes(_ctx(), sha_codigo=SHA_CODIGO)
assert PART and PART['ok'], 'Partições não conferem com o MANIFEST (veja ERRO acima).'

In [ ]:
apresentacao.avisar_revisao_pesos(_ctx())
RES_PESOS = operacoes.executar(operacoes.OP_PESOS, _ctx())
assert RES_PESOS, 'Pesos indisponíveis (veja ERRO acima).'

## 6. Token do Hugging Face Hub (Colab Secrets)

Prepara o **backup** da execução num repositório **privado** do HF Hub (`angeloalsf/reclin-pt-pairawarepooler-biobertpt-seed42`). É camada **extra**: o Drive continua sendo o destino principal. O envio é feito **ao final**, na seção 11, e só com a sua confirmação explícita (o `treinar.py` atual não envia nada durante o treino).

O token nunca é colado em texto plano aqui; vem dos **Secrets** do Colab, e **não é exportado** para o ambiente nesta célula.

1. Crie um Access Token do tipo **Write** em <https://huggingface.co/settings/tokens>
2. No Colab: ícone de **chave** no painel esquerdo → **+ Add new secret** → nome `HF_TOKEN` → cole o valor → ligue **Notebook access**
3. Rode a célula abaixo

Sem o secret o treino roda igual; só o backup extra fica de fora.

In [ ]:
from google.colab import userdata

HF_USER = 'angeloalsf'
MODELO = 'biobertpt'

# A semente é a da fase, vem da constante do protocolo (não de nome de pasta).
SEED, _ = apresentacao.semente_da_fase()
HF_BACKUP_REPO = f'{HF_USER}/reclin-pt-pairawarepooler-{MODELO}-seed{SEED}'

try:
    cred = segredos.obter(segredos.HUGGING_FACE)
except Exception as erro:
    cred = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)

HF_OK = False
if cred:
    try:
        quem = segredos.verificar_escrita_hf(cred)        # consulta somente leitura (whoami); nada é enviado
        HF_OK = True
        print('HF_TOKEN carregado dos Secrets do Colab; token de escrita valido (usuario %s).' % quem['usuario'])
        print('(o valor nunca e impresso, nem exportado para o ambiente nesta celula).')
        print('Backup (secao 11) ->', HF_BACKUP_REPO, '(repositorio PRIVADO)')
        print('Semente da fase:', SEED)
    except Exception as erro:
        print('HF_TOKEN presente, mas nao validou como token de escrita (%s).' % type(erro).__name__)
cred = None
if not HF_OK:
    print('BACKUP NO HF HUB DESLIGADO nesta execucao.')
    print('  1. Crie um token WRITE em https://huggingface.co/settings/tokens')
    print('  2. Colab: icone de chave (Secrets) no painel esquerdo >')
    print('     "+ Add new secret" > nome HF_TOKEN > cole o valor >')
    print('     ligue "Notebook access"')
    print('  3. Rode esta celula de novo.')
    print('  O treino roda normalmente; so o backup extra fica de fora.')

## 7. Treinar a estratégia `pair_aware_pooler` (com retomada)

Encoder `pucpr/biobertpt-all`, marcadores `[E1] [/E1] [E2] [/E2]` no texto de entrada (como em todas as estratégias), 3 classes, `class_weight=balanced`, cabeça `[pooler(h_cls) ; h_[E1] ; h_[E2]]` → Dropout → Linear(3d, 3) (`head_dropout=None` usa o dropout do encoder). **Todos** os pares candidatos entram no treino e no DEV; **3 épocas**, `batch_size=64`, sem parada antecipada. Melhor época pelo macro-F1 do **DEV completo**; o **TEST não é usado**.

Os parâmetros abaixo são os do Baseline e da Pair-Aware linear. **Não os altere**: a ablação só vale se a única diferença entre as quatro células for a leitura dos marcadores e o uso do pooler. A célula confere isso antes de treinar.

A célula escolhe sozinha o que fazer: **treino novo** se não há execução desta configuração, **retomada** se há uma interrompida compatível e, se já foi concluída, **não treina de novo** (nada é sobrescrito). Se o runtime cair, rode as seções 1–6 e esta célula de novo.

No `treino.json` ficam registrados os marcadores ausentes depois da tokenização e, na variante com pooler, se os pesos do pooler vieram do checkpoint pré-treinado (`descricao.cabeca.pooler_do_checkpoint`).

In [ ]:
PARAMETROS = {                 # os do Baseline e da Pair-Aware linear (não alterar: é uma ablação)
    'max_gap': 25, 'ctx_chars': 128, 'max_length': 128, 'epochs': 3, 'batch_size': 64, 'lr': 2e-5,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'class_weight': 'balanced',
    'head_dropout': None,          # None = dropout do encoder
}
NOVA_EXECUCAO = False          # True só para outra tentativa ou parâmetros diferentes do protocolo (exploratória)
MOTIVO = ''                    # opcional (vazio grava "sem motivo informado")

ctx = _ctx(parametros=PARAMETROS, motivo=MOTIVO.strip() or 'Ablação do pooler (Etapa 5, M3): fatorial marcadores × pooler')
from reclin.estrategias import pair_aware as _pa
_cfg_abl = operacoes.config_da_operacao(ctx.espec, PARAMETROS).como_dict()
_cfg_pa = _pa.ConfigPairAware(encoder=_cfg_abl['encoder'], seed=_cfg_abl['seed']).como_dict()
DIFERENCAS = {k: (_cfg_pa.get(k), v) for k, v in _cfg_abl.items() if k != 'tipo' and _cfg_pa.get(k) != v}
assert not DIFERENCAS, f'PARAMETROS difere da Pair-Aware linear do protocolo: {DIFERENCAS} — a ablação exige os mesmos hiperparâmetros.'
protocolo = operacoes.config_da_operacao(ctx.espec, PARAMETROS) == ctx.espec.config_protocolo()
rec = apresentacao.recomendar_ramo(ctx)
if NOVA_EXECUCAO and rec['recomendada'] == operacoes.OP_RETOMAR and not MOTIVO.strip():
    raise SystemExit(f"Há execução interrompida COMPATÍVEL ({', '.join(rec['compativeis'])}). Para RETOMAR, ponha "
                     "NOVA_EXECUCAO = False. Para abandoná-la e iniciar OUTRA tentativa, preencha MOTIVO.")
OPERACAO = operacoes.OP_NOVA if NOVA_EXECUCAO else rec['recomendada']
print('\nOperação:', OPERACAO, '| natureza: EXPLORATÓRIA (ablação posterior ao protocolo da fase: sem nome oficial, não elegível ao TEST)')
print('(o quadro abaixo pode escrever «PROTOCOLO» por comparar com os padrões da classe; a estratégia pair_aware_pooler não consta do protocolo)')
if OPERACAO == operacoes.OP_NOVA and not NOVA_EXECUCAO:
    print('\nNada foi treinado:', rec['motivo'])
    print('Para outra tentativa ou parâmetros diferentes do protocolo (exploratória), ponha NOVA_EXECUCAO = True.')
else:
    QUADRO = apresentacao.quadro_pre_treino(ctx, OPERACAO, sha_codigo=SHA_CODIGO, repositorio=f'angeloalsf/{REPO_NAME}')
    try:
        apresentacao.exigir_quadro(QUADRO, ctx, OPERACAO)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None
    RESULTADO_TREINO = operacoes.executar(OPERACAO, ctx)

## 8. Resultados (métricas + matriz de confusão)

Da execução mais recente (ela fica **selecionada** para as seções 10 e 11; o nome e a natureza — protocolo ou exploratória — são impressos), **só DEV**. O TEST não é lido nem avaliado aqui.

In [ ]:
from reclin.experimento import proveniencia

CTX = _ctx_exec(restaurar=True)
EXECUCAO_SELECIONADA = CTX.execucao            # as seções 10 e 11 enviam SÓ esta execução
_reg = proveniencia.ler(apresentacao.pasta_local_da_execucao(CTX))
_base = CTX.espec.config_protocolo().como_dict()
NATUREZA = 'EXPLORATÓRIA'            # pair_aware_pooler é uma ablação posterior ao protocolo pré-registrado da fase
assert _reg['estrategia'] == 'pair_aware_pooler', 'A execução selecionada não é da estratégia pair_aware_pooler!'
print(f"Execução selecionada: {EXECUCAO_SELECIONADA} | natureza: {NATUREZA} | estado: {_reg['estado']}")
operacoes.executar(operacoes.OP_DEV, CTX)

### 8.1 Checkpoint selecionado: métricas por classe e proveniência

Critério de seleção (global, o do protocolo): **maior macro-F1 do DEV completo** (as 3 classes pesam igual; `negation_of` sozinho não decide), empate → a primeira época. As métricas abaixo vêm do `predicoes_dev.json` do checkpoint selecionado (o DEV completo, 19.064 candidatos; o que a Pair-Aware não pontua vira `no_relation`).

Por época, o `treino.json` guarda apenas `train_loss`, `dev_loss`, `dev_macro_f1` e `dev_negation_of_f1`. P/R por classe a cada época **não** são gravados (exigiria mudar o laço de treino compartilhado); só o checkpoint selecionado tem o detalhamento completo.

In [ ]:
from reclin.experimento import comparacao_dev
from reclin.execucao import predicoes as _pred

import json
PASTA = apresentacao.pasta_local_da_execucao(CTX)
_trn = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
_cfg = json.load(open(PASTA / 'config.json', encoding='utf-8'))
_dev = _pred.ler(PASTA / 'predicoes_dev.json')
_l = comparacao_dev.linha(EXECUCAO_SELECIONADA, _dev)

_ec = (_reg.get('chave_configuracao') or {}).get('estrategia_config') or {}
print('Critério de seleção :', _ec.get('selecao_melhor_epoca') or 'dev_macro_f1 no DEV completo (protocolo)')
print('Melhor época        :', _trn['melhor_epoca'], '| dev_macro_f1 =', round(_trn['melhor_dev_macro_f1'], 4))
print('\nPor época (DEV completo):')
for e in _trn['dev_history']:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in e.items()})
print(f"\nDEV completo (n = {_l['n']}) no checkpoint selecionado:  macro-F1 {_l['macro_f1']:.4f} | acurácia {_l['acuracia']:.4f} | MCC {_l['mcc']:.4f}")
for r in comparacao_dev.FOCO:
    print(f"  {r:16s} P {_l[r + '.precisao']:.4f}  R {_l[r + '.recall']:.4f}  F1 {_l[r + '.f1']:.4f}  suporte {int(_l[r + '.suporte'])}")
print('Matriz de confusão (linhas = verdadeiro, colunas = predito; negation_of, associated_with, no_relation):')
for lin in _l['matriz_confusao']: print('  ', lin)
_prov = _cfg.get('proveniencia_pesos') or {}
print('\nProveniência: seed', _reg['seed'], '| revisão dos pesos', _prov.get('commit') or _reg['pesos_pretreinados'].get('commit'),
      '| hash do manifesto dos pesos', _prov.get('hash_manifesto') or _reg['pesos_pretreinados'].get('hash_manifesto'))
print('  código:', {k: _reg['codigo'].get(k) for k in ('origem', 'revisao')}, '| identidade:', _reg['codigo'].get('identidade_global'))
print('  sha256 do config.json:', _reg.get('sha256_config'), '| natureza:', NATUREZA)
print('  configuração efetiva:', _reg['parametros_de_treinamento'])
print('  conjuntos/versões de dados:', _cfg.get('versoes_dados'))
print('  filtro lexical / léxico:', _ec.get('filtro_lexical'), '/', _ec.get('lexico'))
print('  marcadores ausentes:', (_trn.get('descricao') or {}).get('marcadores_ausentes'))
print('  cabeça:', ((_trn.get('descricao') or {}).get('cabeca') or {}).get('arquitetura'), '| pooler veio do checkpoint pré-treinado?',
      ((_trn.get('descricao') or {}).get('cabeca') or {}).get('pooler_do_checkpoint'))

### 8.2 Comparação no DEV (universo comum)

Compara esta execução com o **Baseline** (`baseline_biobertpt_seed42`) e a **Pair-Aware linear** (`pairaware_biobertpt_seed42`), as duas células já treinadas do fatorial, procurando os `predicoes_dev.json` no disco local e no Drive. Se a outra execução da ablação já estiver no Drive, acrescente o nome da pasta dela à lista. A comparação só é feita entre sidecars com o **mesmo `y_true`**; execuções não encontradas são listadas como tal — **nenhum valor é inventado**. Só DEV; o TEST não é lido.

In [ ]:
from reclin.experimento import persistencia

NOMES_COMPARACAO = [                 # nomes reais das pastas em MyDrive/RECLin-PT/execucoes
    'baseline_biobertpt_seed42',         # só o [CLS], com pooler
    'pairaware_biobertpt_seed42',        # [CLS] + marcadores, sem pooler
    EXECUCAO_SELECIONADA,
]
try:
    RAIZES = [RAIZ_LOCAL, persistencia.raiz_das_execucoes(RAIZ_DRIVE)]
    _cmp = comparacao_dev.carregar_e_comparar(RAIZES, NOMES_COMPARACAO)
    print(comparacao_dev.formatar(_cmp))
except comparacao_dev.ErroComparacao as erro:
    print('COMPARAÇÃO RECUSADA:', erro)

### 8.3 Efeitos da ablação: diferenças pareadas no DEV

Diferença **esta execução − referência** nos mesmos 19.064 candidatos do DEV, para o F1 de `negation_of`, o de `associated_with` e o macro-F1: **bootstrap pareado** (10.000 reamostras, seed 42) **por candidato** e **por documento** (`reclin.analises.auditoria_execucoes.comparar_pareado`, o mesmo cálculo da auditoria) e **McNemar exato**. "Detectada" = o IC95% exclui o zero. Leitura **descritiva e exploratória**, de uma única semente: uma diferença não detectada não prova que o efeito é nulo. A análise definitiva é refeita fora do notebook, com `diagnosticos/auditar_execucoes.py`, depois que os resultados forem enviados à branch de resultados. Só o DEV é lido.

In [ ]:
from reclin import tarefa as _tarefa
from reclin.analises import auditoria_execucoes as _aud
from reclin.util.io import ler_jsonl

REFERENCIAS = [
    ('baseline_biobertpt_seed42', 'efeito dos MARCADORES, com pooler (pair_aware_pooler − Baseline)'),
    ('pairaware_biobertpt_seed42', 'efeito de MANTER O POOLER, com marcadores (pair_aware_pooler − Pair-Aware; negativo = retirar o pooler ajuda)'),
]
_c_te = comparacao_dev.localizar(RAIZES, EXECUCAO_SELECIONADA)
_te = _pred.ler(_c_te)
# documento de cada candidato (mesma ordem do DEV reconstruído da partição congelada; só o DEV é lido)
_cands = _tarefa.candidatos(list(ler_jsonl(Path(REPO) / 'codigo' / 'dados' / 'particoes-v2' / 'dev.jsonl')))
assert _tarefa.y_true(_cands) == list(_te['y_true']), 'o DEV reconstruído não confere com o y_true das predições'
_grupos = [c['doc_id'] for c in _cands]
for _nome, _efeito in REFERENCIAS:
    _c_ref = comparacao_dev.localizar(RAIZES, _nome)
    if _c_ref is None:
        print(f'\n{_nome}: predições não encontradas — nada foi calculado.')
        continue
    _ref = _pred.ler(_c_ref)
    assert _pred.mesmo_conjunto(_ref, _te), 'y_true/labels diferem: as execuções não são comparáveis'
    _r = _aud.comparar_pareado(_te['y_true'], _te['y_pred'], _ref['y_pred'], grupos=_grupos)
    print(f'\n{_efeito}')
    for _k in _aud.METRICAS_COMPARADAS:
        _c, _d = _r['por_candidato'][_k], _r['por_documento'][_k]
        print(f"  {_k:20s} {_c['diferenca']:+.4f} | cand [{_c['ic95'][0]:+.4f}; {_c['ic95'][1]:+.4f}] ({_c['diferenca_na_seed42']})"
              f" | doc [{_d['ic95'][0]:+.4f}; {_d['ic95'][1]:+.4f}] ({_d['diferenca_na_seed42']})")
    print('  McNemar exato (acerto por candidato):', _r['mcnemar'])

## 9. Curvas de treino/validação

De `dev_history`: loss de treino vs. validação (overfitting) e F1 no dev por época (seleção de época). Na Pair-Aware, `dev_macro_f1` é o do DEV completo, o critério de escolha da época.

In [ ]:
import json
import matplotlib.pyplot as plt
PASTA = apresentacao.pasta_local_da_execucao(CTX)
r = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
h = r['dev_history']
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
ax[0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
ax[0].set_xlabel('época'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [e['dev_macro_f1'] for e in h], 'o-', label='dev_macro_f1')
ax[1].plot(ep, [e['dev_negation_of_f1'] for e in h], 's-', label='dev_negation_of_f1')
ax[1].set_xlabel('época'); ax[1].set_ylabel('F1'); ax[1].set_title('F1 no dev'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Salvar resultados + predições no GitHub

Forçamos o `add` (`-f`) de cinco arquivos **da execução selecionada na seção 8** (nenhuma outra), em `codigo/resultados/execucoes/<nome>/`: `config.json`, `treino.json`, `experimento.json`, `predicoes_dev.json` (predições do dev na melhor época) e `metricas.json` (**só DEV**). Depois `commit`, `pull --rebase --autostash` e `push`, como no legado.

Nunca vão para o GitHub: `ultimo.pt`, checkpoints, pesos, logs nem artefatos do TEST. O envio exige execução concluída, `metricas.json` sem a chave `test` e predições sem texto clínico, e vai para uma **branch própria** (`main` e `master` são recusadas; o merge para a main é manual, fora do notebook).

**Nada é enviado até você mudar `PUBLICAR_NO_GITHUB` para `True`.** Antes, a célula mostra exatamente o que seria enviado.

In [ ]:
from reclin.experimento import versionamento

PUBLICAR_NO_GITHUB = False          # True = confirmo enviar estes arquivos do DEV ao GitHub
BRANCH_RESULTADOS = 'resultados-colab'   # fixa (main e master são recusadas)

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA)
PASTA = apresentacao.pasta_local_da_execucao(CTX)
print('Execução a enviar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
try:
    versionamento.conferir(PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO)
    if not PUBLICAR_NO_GITHUB:
        print('\nNada foi enviado. Confira acima; para enviar, ponha PUBLICAR_NO_GITHUB = True e rode de novo.')
    else:
        _cred = segredos.Credencial('GITHUB_PAT', userdata.get('GITHUB_PAT'), 'Colab Secrets')
        try:
            RES_GIT = versionamento.salvar(
                PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO,
                clone='/content/reclin_versionamento', nome_git=GIT_USER_NAME, email_git=GIT_USER_EMAIL, credencial=_cred)
            print('Push concluído (se houve mudanças).')
        except versionamento.ErroVersionamento as erro:
            print('ERRO:', segredos.limpar(str(erro), _cred))
        finally:
            _cred = None
except versionamento.ErroVersionamento as erro:
    print('ERRO:', erro)

## 11. Backup privado do melhor modelo no HF Hub (confirmação explícita)

Publica a execução selecionada na seção 8 (`melhor_modelo/` + manifesto com SHA-256, **sem** `ultimo.pt`) no repositório **privado** `HF_BACKUP_REPO` (seção 6), num prefixo próprio da execução; nada de execuções anteriores é sobrescrito. Use também como **rede de segurança** se a rede cair no meio de um envio: rode de novo.

A célula primeiro mostra o plano (o que seria enviado) e **só envia** com `PUBLICAR_NO_HF = True`. O `HF_TOKEN` é lido dos Secrets **só durante o envio** e removido do ambiente ao terminar.

In [ ]:
PUBLICAR_NO_HF = False              # True = confirmo publicar a execução no repositório PRIVADO HF_BACKUP_REPO

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA, repositorio_hf=HF_BACKUP_REPO)
CTX.confirmacao = operacoes.frase_de_publicacao(CTX.execucao, HF_BACKUP_REPO) if PUBLICAR_NO_HF else ''
print('Execução a publicar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
operacoes.executar(operacoes.OP_PUBLICACAO, CTX)             # plano: nada é enviado
if not PUBLICAR_NO_HF:
    print('\nNada foi enviado. Para publicar, ponha PUBLICAR_NO_HF = True e rode de novo.')
else:
    try:
        with apresentacao.segredo_no_ambiente(segredos.HUGGING_FACE):
            RES_PUBLICACAO = operacoes.executar(operacoes.OP_PUBLICAR, CTX)
        print('best_model em https://huggingface.co/' + HF_BACKUP_REPO)
    except segredos.ErroSegredo as erro:
        print('ERRO:', erro)
        print(apresentacao.instrucoes_secret(segredos.HUGGING_FACE))